### Model Pipeline

1. Raw table_data (string)

        ↓ linearize

2. Linearized text (string)

        ↓ tokenizer.encode()

3. Token IDs (integers)

        ↓ model's embedding layer (learned lookup table, inside the model)

4. Token embeddings + positional embeddings

        ↓ encoder self-attention layers (bidirectional)

5. Encoder hidden states (contextualized representations, one vector per input token)

        ↓ fed into every decoder layer via cross-attention

6. Decoder (causal self-attention + cross-attention into step 5)

        ↓ generates one token at a time, autoregressively

7. Output logits → softmax → generated token

        ↓ repeat step 6-7 until <eos>
        
8. Decoded text = generated financial commentary

### Installs and Imports

In [1]:
!pip install -q  -U transformers
!pip install -q -U datasets
!pip install -q -U evaluate
!pip install -q tokenizers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00


In [2]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch

In [3]:
from datasets import Dataset, DatasetDict
import evaluate


In [4]:
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)

In [5]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cpu
CUDA available: False


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Using preprocessing notebook to use the same variables and features.

In [1]:
import os

print(os.listdir("/content/drive/MyDrive/Colab Notebooks/266/266_final_project/"))

before = set(globals().keys())

['train.json', '266_git_workflow.ipynb', 'train_split.json', 'test_split.json', '266_final_project_Text_analysis_pb.ipynb', 'train_splitEDA.py', 'train_splitEDA.ipynb', 'model1_encoder-decoder.ipynb']


In [2]:
%%capture
%run -i "/content/drive/MyDrive/Colab Notebooks/266/266_final_project/train_splitEDA.ipynb"

In [3]:
after = set(globals().keys())
new_vars = after - before
#variables from exported notebook above
print(new_vars)

{'_i3', 'T5Tokenizer', 'torch', 'Dataset', 'test_df', 'StringIO', 'DataCollatorForSeq2Seq', 'parsed', 'build_encoder_input', 'linearize_table_compact', '_i2', 'sample', 'save_dir', 'train_df', 'drive', 'random', 'evaluate', 'Seq2SeqTrainer', 'summary', 'EarlyStoppingCallback', 'json', 'summarize_market_table', 'DatasetDict', 'token_length', 'tokenizer', 'sample_df', 'linearize_table', 'test_file', '_exit_code', 'pd', 're', 'train_file', 'i', 'np', 'model_name', 'before', 'Seq2SeqTrainingArguments', 'T5ForConditionalGeneration'}


In [4]:
train_df["market_summary"] = train_df["table_data"].apply(summarize_market_table)
test_df["market_summary"] = test_df["table_data"].apply(summarize_market_table)
train_df["decoder_target"] = train_df["report"]
test_df["decoder_target"] = test_df["report"]

#### Build Encoder input
- Encoder processes the structured financial data withfull context, no generation order needed.
- Decoder generates text token by token
- Cross-attention lets the decoder ground each generated word back in the specific numbers/facts from the encoder. This will be helpful to check the factual consistency of hypothesis.
- Bidirectional encoder:The encoder will use full, non-causal self-attention. It needs the full self attention to see the entire table at once to build a complete representation.
- So, no masking the tokens on the input side.
- Decoder: The decoder will use causal masked self-attention since it generates autoregressively and can't peek at future output tokens.
- Models to use : **T5/ BART**



In [5]:
def build_encoder_input(row):

    return (
        "Generate a financial market report.\n\n"
        f"Instruction:\n"
        f"{row['instruction']}\n\n"
        f"Market summary:\n"
        f"{row['market_summary']}"
    )

In [6]:
train_df["encoder_input"] = train_df.apply(build_encoder_input,axis=1)
test_df["encoder_input"] = test_df.apply(build_encoder_input,axis=1)

In [7]:
train_df["input_length"] = train_df["encoder_input"].apply(token_length)
print(train_df["input_length"].describe(percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]))

count    3143.000000
mean      174.361120
std         6.731297
min       148.000000
50%       176.000000
75%       178.000000
90%       181.000000
95%       183.000000
99%       185.000000
max       188.000000
Name: input_length, dtype: float64


In [8]:
train_df["market_summary"] = train_df["table_data"].apply(summarize_market_table)
test_df["market_summary"] = test_df["table_data"].apply(summarize_market_table)

In [9]:
#  inspect
print(train_df.iloc[0]["encoder_input"][:3000])
print("\nTARGET:\n")
print(train_df.iloc[0]["decoder_target"])

Generate a financial market report.

Instruction:
Please act as an expert financial market analyst. Please generate a market report:
1. by analyzing the historical market data provided.
2. following the market report example provided.

Market summary:
Product: Live Cattle Future (front month) (December)
Symbol: LEZ1
Period start: 2021-11-02
Period end: 2021-12-02
Trading days: 132
Starting close: 129.95
Ending close: 170.90
Period high: 171.00
Period low: 128.88
Absolute price change: 40.95
Percentage price change: 31.51%
Daily return volatility: 1.25%
Maximum daily gain: 10.11%
Maximum daily loss: -4.69%
Average volume: 9696
Maximum volume: 31903
Minimum volume: 558
Overall price trend: Upward

TARGET:

Cattle futures posted moderate to strong gains as cash trade stayed supportive in the live cattle market. Dec live cattle gained 1.650 to 137.650, and Feb cattle were .975 higher to 139.575. Feeders saw mixed, to mostly higher market as Jan feeders were slightly lower, losing .050 to 1

In [10]:
print("Targets > 128:",(train_df["target_length"] > 128).mean())
print("Targets > 256:",(train_df["target_length"] > 256).mean())

Targets > 128: 0.42666242443525293
Targets > 256: 0.10054088450524976


In [11]:
print("=" * 80)
print("ENCODER INPUT")
print("=" * 80)

print(train_df.iloc[0]["encoder_input"])

print("\n")
print("=" * 80)
print("DECODER TARGET")
print("=" * 80)

print(train_df.iloc[0]["decoder_target"])

ENCODER INPUT
Generate a financial market report.

Instruction:
Please act as an expert financial market analyst. Please generate a market report:
1. by analyzing the historical market data provided.
2. following the market report example provided.

Market summary:
Product: Live Cattle Future (front month) (December)
Symbol: LEZ1
Period start: 2021-11-02
Period end: 2021-12-02
Trading days: 132
Starting close: 129.95
Ending close: 170.90
Period high: 171.00
Period low: 128.88
Absolute price change: 40.95
Percentage price change: 31.51%
Daily return volatility: 1.25%
Maximum daily gain: 10.11%
Maximum daily loss: -4.69%
Average volume: 9696
Maximum volume: 31903
Minimum volume: 558
Overall price trend: Upward


DECODER TARGET
Cattle futures posted moderate to strong gains as cash trade stayed supportive in the live cattle market. Dec live cattle gained 1.650 to 137.650, and Feb cattle were .975 higher to 139.575. Feeders saw mixed, to mostly higher market as Jan feeders were slightly lo

### Model
1. Fine-tuned pretrained T5-small - Load T5 model : "t5-small"  - Chunk long sequences. Split long documents into 512‑token chunks and process them sequentially.
2. t5 base
2. t5_tokenizer = AutoTokenizer.from_pretrained("google-t5/t5-large")
3. t5_model = AutoModelForSeq2SeqLM.from_pretrained("google-t5/t5-large")
4. LongT5 (4k–16k tokens) - not possible due to cost constraints

In [12]:
# calculate length
train_df["input_length"] = train_df["encoder_input"].apply(token_length)
train_df["target_length"] = train_df["decoder_target"].apply(token_length)

In [13]:
print("Targets > 384:",(train_df["target_length"] > 384).sum())
print("Percentage > 384:",(train_df["target_length"] > 384).mean())

Targets > 384: 42
Percentage > 384: 0.013363028953229399


In [14]:
print(train_df[["encoder_input","decoder_target"]].head())

                                       encoder_input  \
0  Generate a financial market report.\n\nInstruc...   
1  Generate a financial market report.\n\nInstruc...   
2  Generate a financial market report.\n\nInstruc...   
3  Generate a financial market report.\n\nInstruc...   
4  Generate a financial market report.\n\nInstruc...   

                                      decoder_target  
0  Cattle futures posted moderate to strong gains...  
1  Cattle prices are trying to turn higher, as th...  
2  Cattle prices saw price weakness to start the ...  
3  Led by a strong overall commodity market and r...  
4  Feb cattle slipped 0.225 to 138.325, and Apr c...  


#### Convert Pandas DataFrames into Hugging Face datasets

wrap train_df/test_df in a HuggingFace Dataset, apply tokenize_batch, and fine-tune with Seq2SeqTrainer

Dataset + Seq2SeqTrainer training loop

In [15]:
train_dataset = Dataset.from_pandas(train_df[["encoder_input", "decoder_target"]],
                                    preserve_index=False)

test_dataset = Dataset.from_pandas(test_df[["encoder_input", "decoder_target"]],
                                   preserve_index=False)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['encoder_input', 'decoder_target'],
    num_rows: 3143
})
Dataset({
    features: ['encoder_input', 'decoder_target'],
    num_rows: 795
})


### Build Model

In [16]:
from transformers import (AutoTokenizer,AutoModelForSeq2SeqLM)

model1_name = "google-t5/t5-small"

tokenizer = AutoTokenizer.from_pretrained(model1_name)

model1 = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

WE're not using padding="max_length" to ensure that padded label tokens dont contribute to the loss. The data collator will dynamically pad each batch.

In [17]:
Max_input_length = 256
Max_target_length = 384

def tokenize_batch(examples):
    model_inputs = tokenizer(
        examples["encoder_input"],
        max_length=Max_input_length,
        truncation=True
        # padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["decoder_target"],
        max_length=Max_target_length,
        truncation=True
        # padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

### Tokenize
Apply to both train and test datasets : We can see input_ids, attention_mask, labels

In [18]:
tokenized_train = train_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=[
        "encoder_input",
        "decoder_target"])

Map:   0%|          | 0/3143 [00:00<?, ? examples/s]

In [20]:
tokenized_test = test_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=[
        "encoder_input",
        "decoder_target"])

Map:   0%|          | 0/795 [00:00<?, ? examples/s]

In [23]:
# check
print(tokenized_train)
print(tokenized_train[0].keys())

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3143
})
dict_keys(['input_ids', 'attention_mask', 'labels'])


### Create the data collator
Instead of manually padding every example to max_input_length, data collater will handle the padding dynamically per batch

In [24]:
# from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model1)
    # padding="longest"

### Training arguments and Trainer

In [25]:
# from transformers import Seq2SeqTrainingArguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-small-financial-commentary",

    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    num_train_epochs=5,
    predict_with_generate=True,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True, # for T4 GPU
    report_to="none"
)

Create the trainer

In [26]:
# from transformers import Seq2SeqTrainer
trainer = Seq2SeqTrainer(
    model=model1,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    # tokenizer=tokenizer,
    processing_class=tokenizer,
    data_collator=data_collator)

In [27]:
print(type(tokenized_train))
print(type(tokenized_test))

<class 'datasets.arrow_dataset.Dataset'>
<class 'datasets.arrow_dataset.Dataset'>


### Model Training

In [28]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss
1,4.125747,3.754681
2,3.846411,3.592062
3,3.752264,3.515673
4,3.740379,3.482074
5,3.726977,3.469377


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [29]:
# save model
final_model_dir = os.path.join(save_dir, "t5_small_model1")

trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

print(f"Saved model to:{final_model_dir}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to:/content/drive/MyDrive/Colab Notebooks/266/266_final_project/t5_small_model1


In [30]:
# save training metrics
metrics = train_result.metrics

with open(os.path.join(final_model_dir, "training_metrics.json"),"w") as f:
    json.dump(metrics, f, indent=2)

In [31]:
# evaluate on test set
test_metrics = trainer.evaluate(
    max_length=Max_target_length,
    num_beams=4
)

print(test_metrics)

Training Loss,Validation Loss,Epoch
3.726977,3.469377,5


{'eval_loss': 3.469377040863037}


### Report Generation

### Evaluation
- Output commentary Vs. Reference commentary
- Output commentary vs. Underlying numerical facts

- Evaluation Metrics
  - Lexical
  - ROUGE
  - BLEU

- Semantic
  - BERTScore

- Factual grounding
  - LLM as a judge